In [ ]:
# PyTorch core
import torch
import torch.nn as nn
import torch.optim as optim

# Data handling (already used in task a, but safe to include)
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Utilities
import pandas as pd
import itertools
import matplotlib.pyplot as plt

### Data import

In [3]:
IMG_SIZE = (64, 64) #IMG_SIZE = (128, 128)
BATCH_SIZE = 32

train_dir = "icosimal_img_class_03/data_uniform_224_224_sets/train"
val_dir = "icosimal_img_class_03/data_uniform_224_224_sets/validate"

# Transforms
train_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor()
])

val_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor()
])

# Datasets
train_dataset = datasets.ImageFolder(root=train_dir, transform=train_transform)
val_dataset = datasets.ImageFolder(root=val_dir, transform=val_transform)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Class names
class_names = train_dataset.classes
num_classes = len(class_names)

print("Classes:", class_names)
print("Number of classes:", num_classes)

Classes: ['cat', 'chicken', 'cow', 'dog', 'elephant', 'horse', 'rabbit', 'sheep', 'squirrel', 'zebra']
Number of classes: 10


### a) Start from simple CNN architectures and progressively increase their complexity to show the benefit of depth

In [4]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

In [5]:
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

In [6]:
class CNN_1(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((4, 4)),
            nn.Flatten(),
            nn.Linear(16 * 4 * 4, num_classes)
        )

#        self.classifier = nn.Sequential(
#            nn.Flatten(),
#            nn.Linear(16 * 64 * 64, num_classes)
#        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [7]:
class CNN_2(nn.Module):
    def __init__(self, num_classes, dropout_rate=0.0):
        super().__init__()
        
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((4, 4)),
            nn.Flatten(),
            nn.Linear(32 * 4 * 4, 128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, num_classes)
        )

#        self.classifier = nn.Sequential(
#            nn.Flatten(),
#            nn.Linear(32 * 32 * 32, 128),
#            nn.ReLU(),
#            nn.Linear(128, num_classes)
#        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [8]:
class CNN_3(nn.Module):
    def __init__(self, num_classes, dropout_rate=0.0):
        super().__init__()
        
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((4, 4)),
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, num_classes)
        )

#        self.classifier = nn.Sequential(
#            nn.Flatten(),
#            nn.Linear(64 * 16 * 16, 256),
#            nn.ReLU(),
#            nn.Linear(256, num_classes)
#        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [16]:
print("cuda" if torch.cuda.is_available() else "cpu")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

models = {
    "CNN_1": CNN_1(num_classes),
    "CNN_2": CNN_2(num_classes),
    "CNN_3": CNN_3(num_classes)
}
criterion = nn.CrossEntropyLoss()

for name, model in models.items():
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    print(f"\nTraining {name}...")
    for epoch in range(10):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)

        print(f"{name} | Epoch {epoch+1}/10")
        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
        print(f"Val   Loss: {val_loss:.4f}, Val   Acc: {val_acc:.4f}")

cuda

Training CNN_1...
CNN_1 | Epoch 1/10
Train Loss: 2.1895, Train Acc: 0.1954
Val   Loss: 2.1142, Val   Acc: 0.2410
CNN_1 | Epoch 2/10
Train Loss: 2.0612, Train Acc: 0.2690
Val   Loss: 2.0194, Val   Acc: 0.2990
CNN_1 | Epoch 3/10
Train Loss: 1.9846, Train Acc: 0.3160
Val   Loss: 1.9594, Val   Acc: 0.3282
CNN_1 | Epoch 4/10
Train Loss: 1.9262, Train Acc: 0.3378
Val   Loss: 1.9050, Val   Acc: 0.3462
CNN_1 | Epoch 5/10
Train Loss: 1.8862, Train Acc: 0.3533
Val   Loss: 1.8726, Val   Acc: 0.3607
CNN_1 | Epoch 6/10
Train Loss: 1.8524, Train Acc: 0.3640
Val   Loss: 1.8450, Val   Acc: 0.3722
CNN_1 | Epoch 7/10
Train Loss: 1.8241, Train Acc: 0.3737
Val   Loss: 1.8247, Val   Acc: 0.3705
CNN_1 | Epoch 8/10
Train Loss: 1.8028, Train Acc: 0.3818
Val   Loss: 1.8027, Val   Acc: 0.3833
CNN_1 | Epoch 9/10
Train Loss: 1.7831, Train Acc: 0.3902
Val   Loss: 1.7954, Val   Acc: 0.3902
CNN_1 | Epoch 10/10
Train Loss: 1.7651, Train Acc: 0.3940
Val   Loss: 1.7721, Val   Acc: 0.3952

Training CNN_2...
CNN_2 

NEW CODE to replace upper code

In [ ]:
print("cuda" if torch.cuda.is_available() else "cpu")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

models = {
    "CNN_1": CNN_1(num_classes),
    "CNN_2": CNN_2(num_classes),
    "CNN_3": CNN_3(num_classes)
}

criterion = nn.CrossEntropyLoss()

results = []

for name, model in models.items():
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    print(f"\nTraining {name}...")

    best_val_acc = 0

    for epoch in range(10):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)

        best_val_acc = max(best_val_acc, val_acc)

        print(f"{name} | Epoch {epoch+1}/10")
        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
        print(f"Val   Loss: {val_loss:.4f}, Val   Acc: {val_acc:.4f}")

    # store final results
    results.append({
        "model": name,
        "final_train_loss": train_loss,
        "final_train_acc": train_acc,
        "final_val_loss": val_loss,
        "final_val_acc": val_acc,
        "best_val_acc": best_val_acc
    })

# -------------------------
# Create comparison table
# -------------------------
results_df = pd.DataFrame(results)

print("\n=== Model Comparison ===")
display(results_df.sort_values(by="best_val_acc", ascending=False))

cuda

Training CNN_1...
CNN_1 | Epoch 1/10
Train Loss: 2.1874, Train Acc: 0.2030
Val   Loss: 2.0912, Val   Acc: 0.2547
CNN_1 | Epoch 2/10
Train Loss: 2.0373, Train Acc: 0.2878
Val   Loss: 2.0325, Val   Acc: 0.2698
CNN_1 | Epoch 3/10
Train Loss: 1.9701, Train Acc: 0.3159
Val   Loss: 1.9508, Val   Acc: 0.3280
CNN_1 | Epoch 4/10
Train Loss: 1.9211, Train Acc: 0.3343
Val   Loss: 1.9117, Val   Acc: 0.3403
CNN_1 | Epoch 5/10
Train Loss: 1.8857, Train Acc: 0.3513
Val   Loss: 1.8800, Val   Acc: 0.3527
CNN_1 | Epoch 6/10
Train Loss: 1.8479, Train Acc: 0.3675
Val   Loss: 1.8526, Val   Acc: 0.3602
CNN_1 | Epoch 7/10
Train Loss: 1.8162, Train Acc: 0.3754
Val   Loss: 1.8194, Val   Acc: 0.3717
CNN_1 | Epoch 8/10
Train Loss: 1.7879, Train Acc: 0.3878
Val   Loss: 1.7950, Val   Acc: 0.3858
CNN_1 | Epoch 9/10
Train Loss: 1.7657, Train Acc: 0.3943
Val   Loss: 1.7640, Val   Acc: 0.3990
CNN_1 | Epoch 10/10
Train Loss: 1.7438, Train Acc: 0.4044
Val   Loss: 1.7655, Val   Acc: 0.3957

Training CNN_2...
CNN_2 

### b) Show the importance of hyperparameter tuning.

In [ ]:
learning_rates = [0.001, 0.0001]
dropout_rates = [0.0, 0.3, 0.5]
optimizers_list = ["Adam", "RMSprop"]
batch_sizes = [32, 64]

num_epochs = 5

grid = list(itertools.product(
    learning_rates,
    dropout_rates,
    optimizers_list,
    batch_sizes
))

print("Total experiments:", len(grid))
print(grid[:5])  # preview first few combinations

Total experiments: 24
[(0.001, 0.0, 'Adam', 32), (0.001, 0.0, 'Adam', 64), (0.001, 0.0, 'RMSprop', 32), (0.001, 0.0, 'RMSprop', 64), (0.001, 0.3, 'Adam', 32)]


In [ ]:
class CNN_3_Tunable(nn.Module):
    def __init__(self, num_classes, dropout_rate=0.0):
        super().__init__()
        
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((4, 4)),
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, device, num_epochs=5):
    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": []
    }

    for epoch in range(num_epochs):
        # ---- Training ----
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()

        train_loss = running_loss / total
        train_acc = correct / total

        # ---- Validation ----
        model.eval()
        running_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)

                running_loss += loss.item() * images.size(0)
                preds = outputs.argmax(dim=1)
                total += labels.size(0)
                correct += (preds == labels).sum().item()

        val_loss = running_loss / total
        val_acc = correct / total

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(
            f"Epoch {epoch+1}/{num_epochs} | "
            f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}"
        )

    return history


In [ ]:
results = []
all_histories = {}

for i, (lr, dropout_rate, optimizer_name, batch_size) in enumerate(grid, start=1):
    print("\n" + "=" * 70)
    print(f"Experiment {i}/{len(grid)}")
    print(f"learning_rate={lr}, dropout={dropout_rate}, optimizer={optimizer_name}, batch_size={batch_size}")

    model = CNN_3_Tunable(num_classes=num_classes, dropout_rate=dropout_rate).to(device)
    criterion = nn.CrossEntropyLoss()

    if optimizer_name == "Adam":
        optimizer = optim.Adam(model.parameters(), lr=lr)
    elif optimizer_name == "RMSprop":
        optimizer = optim.RMSprop(model.parameters(), lr=lr)
    else:
        raise ValueError(f"Unsupported optimizer: {optimizer_name}")

    history = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
        num_epochs=num_epochs
    )

    config_name = f"lr={lr}_dropout={dropout_rate}_opt={optimizer_name}_bs={batch_size}"
    all_histories[config_name] = history

    results.append({
        "learning_rate": lr,
        "dropout": dropout_rate,
        "optimizer": optimizer_name,
        "batch_size": batch_size,
        "final_train_loss": history["train_loss"][-1],
        "final_train_acc": history["train_acc"][-1],
        "final_val_loss": history["val_loss"][-1],
        "final_val_acc": history["val_acc"][-1],
        "best_val_acc": max(history["val_acc"]),
        "best_val_loss": min(history["val_loss"])
    })

results_df = pd.DataFrame(results)

In [ ]:
# ---- Full sorted result table ----
results_sorted = results_df.sort_values(by="best_val_acc", ascending=False)
display(results_sorted)

# ---- Best configuration ----
best_result = results_sorted.iloc[0]

print("Best configuration:")
display(best_result.to_frame())

# ---- Pivot table / matrix view ----
pivot_table = results_df.pivot_table(
    index="optimizer",
    columns=["learning_rate", "dropout", "batch_size"],
    values="best_val_acc"
)

print("Pivot table of best validation accuracy:")
display(pivot_table)

# ---- Average result by optimizer ----
optimizer_summary = results_df.groupby("optimizer")[["best_val_acc", "final_val_acc", "final_val_loss"]].mean()
print("Average performance per optimizer:")
display(optimizer_summary)

# ---- Save results if needed ----
results_df.to_csv("cnn3_grid_search_results.csv", index=False)
print("Results saved to cnn3_grid_search_results.csv")

### c) Show models that overfit and underfit and explain the reasons for that. 

### d) Experiment with regularization techniques.

In [ ]:
class CNN_3_Regularized(nn.Module):
    def __init__(self, num_classes, dropout_rate=0.0):
        super().__init__()
        
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((4, 4)),
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, device, num_epochs=10):
    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": []
    }

    for epoch in range(num_epochs):
        # Training
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()

        train_loss = running_loss / total
        train_acc = correct / total

        # Validation
        model.eval()
        running_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)

                running_loss += loss.item() * images.size(0)
                preds = outputs.argmax(dim=1)
                total += labels.size(0)
                correct += (preds == labels).sum().item()

        val_loss = running_loss / total
        val_acc = correct / total

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(
            f"Epoch {epoch+1}/{num_epochs} | "
            f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}"
        )

    return history

In [ ]:
regularization_configs = [
    {"name": "Baseline", "dropout": 0.0, "weight_decay": 0.0},
    {"name": "Dropout", "dropout": 0.5, "weight_decay": 0.0},
    {"name": "WeightDecay", "dropout": 0.0, "weight_decay": 1e-4},
    {"name": "Dropout+WeightDecay", "dropout": 0.5, "weight_decay": 1e-4},
]

num_epochs = 10
criterion = nn.CrossEntropyLoss()

results = []
histories = {}

for config in regularization_configs:
    print("\n" + "=" * 60)
    print(f"Running: {config['name']}")
    print(f"dropout={config['dropout']}, weight_decay={config['weight_decay']}")

    model = CNN_3_Regularized(
        num_classes=num_classes,
        dropout_rate=config["dropout"]
    ).to(device)

    optimizer = optim.Adam(
        model.parameters(),
        lr=0.001,
        weight_decay=config["weight_decay"]
    )

    history = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
        num_epochs=num_epochs
    )

    histories[config["name"]] = history

    results.append({
        "config": config["name"],
        "dropout": config["dropout"],
        "weight_decay": config["weight_decay"],
        "final_train_loss": history["train_loss"][-1],
        "final_train_acc": history["train_acc"][-1],
        "final_val_loss": history["val_loss"][-1],
        "final_val_acc": history["val_acc"][-1],
        "best_val_acc": max(history["val_acc"]),
        "best_val_loss": min(history["val_loss"])
    })

results_df = pd.DataFrame(results)

In [ ]:
results_sorted = results_df.sort_values(by="best_val_acc", ascending=False)

print("Regularization comparison:")
display(results_sorted)

In [ ]:
plt.figure(figsize=(10, 6))

for name, history in histories.items():
    plt.plot(history["val_acc"], label=f"{name} - Val Acc")

plt.title("Validation Accuracy for Different Regularization Techniques")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

plt.figure(figsize=(10, 6))

for name, history in histories.items():
    gap = [train - val for train, val in zip(history["train_acc"], history["val_acc"])]
    plt.plot(gap, label=f"{name} - Train/Val Gap")

plt.title("Generalization Gap for Different Regularization Techniques")
plt.xlabel("Epoch")
plt.ylabel("Train Accuracy - Validation Accuracy")
plt.legend()
plt.show()

### e) Experiment with different optimization algorithms (e.g., Adam, RMSprop) and compare their performance.

In [ ]:
optimizer_configs = [
    {"name": "Adam", "lr": 0.001},
    {"name": "RMSprop", "lr": 0.001},
    {"name": "SGD", "lr": 0.01, "momentum": 0.9}
]

num_epochs = 10

In [ ]:
class CNN_3_Optim(nn.Module):
    def __init__(self, num_classes, dropout_rate=0.0):
        super().__init__()
        
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((4, 4)),
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, device, num_epochs=10):
    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": []
    }

    for epoch in range(num_epochs):
        # ---- Training ----
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()

        train_loss = running_loss / total
        train_acc = correct / total

        # ---- Validation ----
        model.eval()
        running_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)

                running_loss += loss.item() * images.size(0)
                preds = outputs.argmax(dim=1)
                total += labels.size(0)
                correct += (preds == labels).sum().item()

        val_loss = running_loss / total
        val_acc = correct / total

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(
            f"Epoch {epoch+1}/{num_epochs} | "
            f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}"
        )

    return history


criterion = nn.CrossEntropyLoss()

results = []
histories = {}

for config in optimizer_configs:
    print("\n" + "=" * 60)
    print(f"Running optimizer: {config['name']}")

    model = CNN_3_Optim(num_classes=num_classes, dropout_rate=0.0).to(device)

    if config["name"] == "Adam":
        optimizer = optim.Adam(model.parameters(), lr=config["lr"])
    elif config["name"] == "RMSprop":
        optimizer = optim.RMSprop(model.parameters(), lr=config["lr"])
    elif config["name"] == "SGD":
        optimizer = optim.SGD(
            model.parameters(),
            lr=config["lr"],
            momentum=config["momentum"]
        )
    else:
        raise ValueError(f"Unsupported optimizer: {config['name']}")

    history = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
        num_epochs=num_epochs
    )

    histories[config["name"]] = history

    results.append({
        "optimizer": config["name"],
        "learning_rate": config["lr"],
        "final_train_loss": history["train_loss"][-1],
        "final_train_acc": history["train_acc"][-1],
        "final_val_loss": history["val_loss"][-1],
        "final_val_acc": history["val_acc"][-1],
        "best_val_acc": max(history["val_acc"]),
        "best_val_loss": min(history["val_loss"])
    })

results_df = pd.DataFrame(results)

In [ ]:
# ---- Sorted result table ----
results_sorted = results_df.sort_values(by="best_val_acc", ascending=False)

print("Optimizer comparison:")
display(results_sorted)

# ---- Best optimizer ----
best_result = results_sorted.iloc[0]

print("Best optimizer configuration:")
display(best_result.to_frame())

In [ ]:
plt.figure(figsize=(10, 6))

for name, history in histories.items():
    plt.plot(history["val_acc"], label=f"{name} - Val Acc")

plt.title("Validation Accuracy for Different Optimizers")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

plt.figure(figsize=(10, 6))

for name, history in histories.items():
    plt.plot(history["train_loss"], label=f"{name} - Train Loss")

plt.title("Training Loss for Different Optimizers")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

plt.figure(figsize=(10, 6))

for name, history in histories.items():
    gap = [train - val for train, val in zip(history["train_acc"], history["val_acc"])]
    plt.plot(gap, label=f"{name} - Train/Val Gap")

plt.title("Generalization Gap for Different Optimizers")
plt.xlabel("Epoch")
plt.ylabel("Train Accuracy - Validation Accuracy")
plt.legend()
plt.show()